# 09 — Dynamo vector-field fitting and quantitative metrics

Consumes the H5AD written by notebook 08b, fits condition-specific vector fields, computes the metrics used by Fig. 6F/Fig. S3E, and writes quantified WT/KO H5AD files.


In [ ]:
# Centralized paths, deterministic seed, and scheduler-aware thread limits.
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "notebooks" / "paths.py").exists() else Path.cwd().parent
if not (repo_root / "notebooks" / "paths.py").exists():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory.")
sys.path.insert(0, str(repo_root / "notebooks"))
from paths import P38_THREADS, RANDOM_SEED, legacy_path, output_path, p38_path


In [1]:
# --- Cell 1: Environment & Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
import anndata
import dynamo as dyn

# Thread limits are set before NumPy import by notebooks/paths.py using P38_THREADS/NSLOTS.

# 2. 忽略非致命警告
warnings.filterwarnings("ignore", category=UserWarning, module='numba')
warnings.filterwarnings("ignore", category=FutureWarning)

# 3. 绘图与种子设置
dyn.configuration.set_figure_params('dynamo', background='white')
np.random.seed(RANDOM_SEED)

# 4. 定义输出路径
output_dir = legacy_path("20260103-p38/pipeline_steps")
os.makedirs(output_dir, exist_ok=True)

print("Environment setup complete.")

# --- Cell 2: Data Loading & Subsetting ---
print("Loading raw data...")
# 请确保路径正确
adata = anndata.read_h5ad(legacy_path("20250526p38-draw/sub/dynamo/velocity_ready_full_with_detailed_index_check.h5ad"))

# 定义细胞类型映射
mapping_table = pd.read_csv(repo_root / "notebooks" / "cell_types.csv", dtype={"cluster": str})
celltype_mapping = dict(zip(mapping_table["cluster"], mapping_table["celltype"]))
adata.obs["RNA_snn_res.0.8"] = adata.obs["RNA_snn_res.0.8"].astype(str)
adata.obs["celltype"] = adata.obs["RNA_snn_res.0.8"].map(celltype_mapping)
if adata.obs["celltype"].isna().any():
    missing_clusters = sorted(adata.obs.loc[adata.obs["celltype"].isna(), "RNA_snn_res.0.8"].unique())
    raise ValueError(f"Unmapped RNA_snn_res.0.8 clusters: {missing_clusters}")

# 提取 pDC 轨迹相关的 Cluster
pdc_trajectory_clusters = [
  "4 CD115+ CDP", "7 CD115- CDP","12 pre-pDC", "0 Mzb1+ pDC", "1 Iglc3+ pDC"
]
adata_pdc = adata[adata.obs['celltype'].isin(pdc_trajectory_clusters)].copy()

print(f"Data loaded. Subsetting complete. Shape: {adata_pdc.shape}")
print(f"Cells per condition: {adata_pdc.obs['orig.ident'].value_counts()}")

# --- Cell 3: Joint Preprocessing & Embedding (The "Gold Standard") ---
print("Starting Joint Preprocessing...")

preprocessor = dyn.preprocessing.Preprocessor()

# 1. 预处理
# recipe='monocle' 在动力学分析中通常表现更好，它会处理 size factor
# 这一步是在 WT+KO 合并数据上做的
preprocessor.preprocess_adata(adata_pdc, recipe='monocle', tkey=None)

# 2. 降维 (PCA + UMAP)
# 这一步生成的 X_umap 将作为后续 WT 和 KO 对比的共同基准地图
print("Running dimensionality reduction (PCA -> UMAP)...")
dyn.tl.reduceDimension(adata_pdc, basis='umap')

# 保存这一步的中间结果，防止崩了重跑
adata_pdc.write_h5ad(os.path.join(output_dir, "step3_joint_embedding.h5ad"))

Environment setup complete.
Loading raw data...
Data loaded. Subsetting complete. Shape: (8176, 20884)
Cells per condition: orig.ident
WT    4694
KO    3482
Name: count, dtype: int64
Starting Joint Preprocessing...
|-----> Running monocle preprocessing pipeline...
|-----------> filtered out 0 outlier cells
|-----------> filtered out 14307 outlier genes
|-----> PCA dimension reduction
|-----> <insert> X_pca to obsm in AnnData Object.
|-----> [Preprocessor-monocle] completed [77.7671s]
Running dimensionality reduction (PCA -> UMAP)...
|-----> retrieve data for non-linear dimension reduction...
|-----? adata already have basis umap. dimension reduction umap will be skipped! 
set enforce=True to re-performing dimension reduction.
|-----> Start computing neighbor graph...
|-----------> X_data is None, fetching or recomputing...
|-----> fetching X data from layer:None, basis:pca
|-----> method arg is None, choosing methods automatically...
|-----------> method pynn selected
|-----> [UMAP] co

In [2]:
# --- Cell 4: Split & Calculate Vector Fields ---
# 提取数据 (此时它们共享 X_umap)
adata_wt = adata_pdc[adata_pdc.obs['orig.ident'] == 'WT'].copy()
adata_ko = adata_pdc[adata_pdc.obs['orig.ident'] == 'KO'].copy()

def run_dynamics_pipeline(adata_subset, label):
    print(f"--- Processing {label} ---")
    
    # 1. Moments: 必须基于 split 后的数据重算邻居关系
    # 这里的 group=None 表示不强制跨 Cluster 平滑，保留局部结构
    dyn.tl.moments(adata_subset, group=None)
    
    # 2. Dynamics: 计算 gamma 等动力学参数
    print(f"[{label}] Calculating dynamics parameters...")
    dyn.tl.dynamics(adata_subset, model='deterministic')
    
    # 3. Cell Velocities: 投影速率到 UMAP
    # 注意：这里不需要再运行 reduceDimension，因为已经继承了 joint UMAP
    print(f"[{label}] Projecting velocities...")
    dyn.tl.cell_velocities(adata_subset, basis='umap')
    
    # 4. Vector Field: 构建连续场 (最耗时的一步)
    print(f"[{label}] Learning Vector Field...")
    dyn.vf.VectorField(adata_subset, basis='umap')
    
    return adata_subset

# 分别运行
adata_wt = run_dynamics_pipeline(adata_wt, "WT")
adata_ko = run_dynamics_pipeline(adata_ko, "KO")

Splitting data for independent dynamics calculation...
--- Processing WT ---
|-----> calculating first/second moments...
|-----? layer X_counts is not in any of the (['X_spliced', 'X_unspliced'], ['X_new', 'X_total'], ['X_uu', 'X_ul', 'X_su', 'X_sl']) groups, skipping...
|-----? layer X_counts is not in any of the (['X_spliced', 'X_unspliced'], ['X_new', 'X_total'], ['X_uu', 'X_ul', 'X_su', 'X_sl']) groups, skipping...
|-----? layer X_counts is not in any of the (['X_spliced', 'X_unspliced'], ['X_new', 'X_total'], ['X_uu', 'X_ul', 'X_su', 'X_sl']) groups, skipping...
|-----> [moments calculation] completed [36.0710s]
[WT] Calculating dynamics parameters...
|-----> dynamics_del_2nd_moments_key is None. Using default value from DynamoAdataConfig: dynamics_del_2nd_moments_key=False


estimating gamma: 100%|██████████████████████████| 2000/2000 [01:42<00:00, 19.58it/s]


[WT] Projecting velocities...
|-----? Some indices in neighbors are larger than the number of observations and thus not valid.
|-----> Neighbor graph is broken, recomputing....
|-----> Start computing neighbor graph...
|-----------> X_data is None, fetching or recomputing...
|-----> fetching X data from layer:None, basis:pca
|-----> method arg is None, choosing methods automatically...
|-----------> method ball_tree selected
|-----> [calculating transition matrix via pearson kernel with sqrt transform.] in progress: 100.0000%|-----> [calculating transition matrix via pearson kernel with sqrt transform.] completed [10.6640s]
|-----> [projecting velocity vector to low dimensional embedding] in progress: 100.0000%|-----> [projecting velocity vector to low dimensional embedding] completed [2.2484s]
|-----> method arg is None, choosing methods automatically...
|-----------> method kd_tree selected
[WT] Learning Vector Field...
|-----> VectorField reconstruction begins...
|-----> Retrieve X 

estimating gamma: 100%|██████████████████████████| 2000/2000 [01:19<00:00, 25.21it/s]


[KO] Projecting velocities...
|-----? Some indices in neighbors are larger than the number of observations and thus not valid.
|-----> Neighbor graph is broken, recomputing....
|-----> Start computing neighbor graph...
|-----------> X_data is None, fetching or recomputing...
|-----> fetching X data from layer:None, basis:pca
|-----> method arg is None, choosing methods automatically...
|-----------> method ball_tree selected
|-----> [calculating transition matrix via pearson kernel with sqrt transform.] in progress: 100.0000%|-----> [calculating transition matrix via pearson kernel with sqrt transform.] completed [8.1987s]
|-----> [projecting velocity vector to low dimensional embedding] in progress: 100.0000%|-----> [projecting velocity vector to low dimensional embedding] completed [1.6497s]
|-----> method arg is None, choosing methods automatically...
|-----------> method kd_tree selected
[KO] Learning Vector Field...
|-----> VectorField reconstruction begins...
|-----> Retrieve X a

In [3]:
# --- Cell 5: Quantitative Metrics (Speed, Curvature, Acceleration) ---

def calculate_quantities(adata_subset, label):
    print(f"--- Calculating Metrics for {label} ---")
    
    # 1. 速度 (Speed)
    print(f"[{label}] Calculating Speed...")
    dyn.vf.speed(adata_subset, basis='umap')
    
    # 2. 加速度 (Acceleration) - 反映驱动力
    print(f"[{label}] Calculating Acceleration...")
    dyn.vf.acceleration(adata_subset, basis='umap')
    
    # 3. 曲率 (Curvature) - 反映分化路径的混乱程度
    print(f"[{label}] Calculating Curvature...")
    dyn.vf.curvature(adata_subset, basis='umap')
    
    # 4. 散度 (Divergence) - 反映源(Source)和汇(Sink)
    print(f"[{label}] Calculating Divergence...")
    dyn.vf.divergence(adata_subset, basis='umap')
    
    return adata_subset

adata_wt = calculate_quantities(adata_wt, "WT")
adata_ko = calculate_quantities(adata_ko, "KO")

--- Calculating Metrics for WT ---
[WT] Calculating Speed...
[WT] Calculating Acceleration...
|-----> [Calculating acceleration] in progress: 100.0000%|-----> [Calculating acceleration] completed [0.1099s]
[WT] Calculating Curvature...
|-----> [Calculating acceleration] in progress: 100.0000%|-----> [Calculating acceleration] completed [0.1049s]
|-----> [Calculating curvature] in progress: 100.0000%|-----> [Calculating curvature] completed [0.1773s]
[WT] Calculating Divergence...


Calculating divergence: 100%|████████████████████████| 5/5 [00:00<00:00, 12.27it/s]


--- Calculating Metrics for KO ---
[KO] Calculating Speed...
[KO] Calculating Acceleration...
|-----> [Calculating acceleration] in progress: 100.0000%|-----> [Calculating acceleration] completed [0.1084s]
[KO] Calculating Curvature...
|-----> [Calculating acceleration] in progress: 100.0000%|-----> [Calculating acceleration] completed [0.1003s]
|-----> [Calculating curvature] in progress: 100.0000%|-----> [Calculating curvature] completed [0.1320s]
[KO] Calculating Divergence...


Calculating divergence: 100%|████████████████████████| 4/4 [00:00<00:00, 16.45it/s]

Quantitative metrics calculated.


In [24]:
# --- Cell 10: Advanced Biophysical Metrics ---
import numpy as np

def calculate_advanced_metrics(adata_subset, label):
    print(f"--- Calculating Advanced Metrics for {label} ---")
    
    # 1. 计算 Coherence (协同性/置信度)
    # 这能直接量化你说的 "混乱 (Chaotic)"
    print(f"[{label}] Calculating Velocity Confidence (Coherence)...")
    dyn.tl.cell_wise_confidence(adata_subset)
    # 结果存储在 adata.obs['jaccard_velocity_confidence'] (或者类似的 key)
    
    # 2. 计算 Jacobian (这一步比较慢，是核心数学计算)
    # 这能量化 "可塑性" 和 "稳定性"
    print(f"[{label}] Calculating Jacobian (Stability)...")
    try:
        # 计算前30个维度的 Jacobian 以节省时间 (通常前几个PC包含了主要动力学)
        dyn.vf.jacobian(adata_subset, regulators=None, effectors=None)
        
        # 提取 Jacobian 的行列式 (Determinant) -> 代表扩张/收缩
        # 提取 Jacobian 的最大特征值实部 -> 代表稳定性 (负值越小越稳定)
        # 注意：Dynamo 的 jacobian 结果通常存在 adata.uns['jacobian'] 中，需要提取到 obs
        
        # 这里我们需要手动提取 summary statistics 到 obs 以便画图
        # 获取每个细胞的 Jacobian 矩阵的性质
        jac_dict = adata_subset.uns['jacobian_pca']['jacobian_gene'] # 或者 'jacobian'
        # 注意：提取 Jacobian 特征比较复杂，Dynamo 有时会自动计算 rank_jacobian
        
        # 简化版：我们使用 dynamo 自带的 curl (旋度) 作为流场复杂度的补充
        dyn.vf.curl(adata_subset, basis='umap')
        
    except Exception as e:
        print(f"Warning: Jacobian calculation specific extraction skipped: {e}")
        # 如果全基因组 Jacobian 太慢，至少算出 Curl
        dyn.vf.curl(adata_subset, basis='umap')

    return adata_subset

# 运行计算
adata_wt = calculate_advanced_metrics(adata_wt, "WT")
adata_ko = calculate_advanced_metrics(adata_ko, "KO")

# --- 检查生成的 Key ---
# 不同的 Dynamo 版本生成的 key 可能不同，通常是 'velocity_confidence'
confidence_key = [k for k in adata_wt.obs.keys() if 'confidence' in k]
print(f"检测到的 Coherence 指标列名: {confidence_key}")

--- Calculating Advanced Metrics for WT ---
[WT] Calculating Velocity Confidence (Coherence)...
[WT] Calculating Jacobian (Stability)...


Calculating 2-D curl: 100%|█████████████████| 4694/4694 [00:00<00:00, 12282.72it/s]


--- Calculating Advanced Metrics for KO ---
[KO] Calculating Velocity Confidence (Coherence)...
[KO] Calculating Jacobian (Stability)...


Calculating 2-D curl: 100%|█████████████████| 3482/3482 [00:00<00:00, 13299.17it/s]

检测到的 Coherence 指标列名: ['jaccard_velocity_confidence']


In [5]:
# --- Cell 7: Save Final Results ---
final_output_dir = legacy_path("20260103-p38/final_results")
os.makedirs(final_output_dir, exist_ok=True)

print("Saving final datasets...")
# 分别保存，不要合并，以免覆盖向量场信息
adata_wt.write_h5ad(os.path.join(final_output_dir, "adata_wt_dynamo_quantified.h5ad"))
adata_ko.write_h5ad(os.path.join(final_output_dir, "adata_ko_dynamo_quantified.h5ad"))

print(f"All done! Files saved to: {final_output_dir}")
print("You can now proceed to specific plotting scripts using these files.")

Saving final datasets...
All done! Files saved to: /data3/Group8/gonglihao/20260103-p38/final_results
You can now proceed to specific plotting scripts using these files.
